# CO_heatmap — Basel Traffic Light Heatmap

Basel Traffic Light heatmap (9 models × 24 assets) after conformal
correction at α = 0.01. Cell labels show annual violation counts
scaled to 250 trading days (Figure 2).

**Output:** `fig_traffic_light.pdf`/`.png`

In [ ]:
"""
Basel Traffic Light heatmap after conformal correction (Figure 2).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Self-contained plotting style — duplicated across figure Quantlets
# by Q² convention.
C_GRAY = '#666666'
C_TEAL = '#00A651'
C_RED = '#E31E24'
C_BLUE = '#0066CC'
C_PURPLE = '#7B2FBE'

plt.rcParams.update({
    'font.family': 'serif', 'axes.grid': False,
    'savefig.transparent': True, 'savefig.dpi': 300,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10,
})

MODEL_ORDER = ['Chronos-Small', 'Chronos-Mini', 'TimesFM-2.5',
               'Moirai-2.0', 'Lag-Llama',
               'GJR-GARCH', 'GARCH-N', 'Hist-Sim', 'EWMA']
MODEL_SHORT = ['Chr-S', 'Chr-M', 'TFM', 'Moirai', 'L-Llama',
               'GJR', 'G-N', 'HS', 'EWMA']

SCRIPT_DIR = Path('.').resolve()
BASE = SCRIPT_DIR.parent
DATA = BASE / 'cfp_ijf_data' / 'paper_outputs' / 'tables'
FIG_DIR = BASE / 'figures'
OUT = SCRIPT_DIR

df = pd.read_csv(DATA / 'all_results.csv')
d01 = df[df['alpha'] == 0.01].copy()

print(f'Loaded {len(df)} rows '
      f'({df["model"].nunique()} models, '
      f'{df["symbol"].nunique()} assets, '
      f'{df["alpha"].nunique()} alphas)')

ASSETS = sorted(d01['symbol'].unique())

tl_map = {'Green': 0, 'Yellow': 1, 'Red': 2}
mat = np.full((len(MODEL_ORDER), len(ASSETS)), np.nan)
for i, m in enumerate(MODEL_ORDER):
    for j, a in enumerate(ASSETS):
        row = d01[(d01['model'] == m) & (d01['symbol'] == a)]
        if len(row) == 1:
            mat[i, j] = tl_map.get(row.iloc[0]['TL_cp'], np.nan)

cmap = ListedColormap([C_TEAL, '#DAA520', C_RED])
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.imshow(mat, cmap=cmap, vmin=0, vmax=2, aspect='auto')
ax.set_title(r'Basel Traffic Light After Conformal Correction ($\alpha = 0.01$)',
             fontsize=12, fontweight='bold', pad=10)

ax.set_xticks(range(len(ASSETS)))
ax.set_xticklabels(ASSETS, fontsize=7, rotation=45, ha='right')
ax.set_yticks(range(len(MODEL_ORDER)))
ax.set_yticklabels(MODEL_SHORT, fontsize=8)

for i in range(len(MODEL_ORDER)):
    for j in range(len(ASSETS)):
        row = d01[(d01['model'] == MODEL_ORDER[i]) &
                  (d01['symbol'] == ASSETS[j])]
        if len(row) == 1:
            v = int(row.iloc[0]['viol_cp'])
            ax.text(j, i, str(v), ha='center', va='center',
                    fontsize=6,
                    color='white' if mat[i, j] == 2 else 'black')

FIG_DIR.mkdir(exist_ok=True)
for ext in ['pdf', 'png']:
    fig.savefig(OUT / f'fig_traffic_light.{ext}', bbox_inches='tight')
    fig.savefig(FIG_DIR / f'fig_traffic_light.{ext}', bbox_inches='tight')

plt.show()
plt.close(fig)
print('Saved: fig_traffic_light.pdf/.png')